In [1]:
import sys
import numpy as np
import pandas as pd

print("PYTHON:", sys.executable)
print("numpy:", np.__version__, np.__file__)
print("pandas:", pd.__version__, pd.__file__)

PYTHON: /root/llm/je/bin/python
numpy: 2.2.6 /root/llm/je/lib/python3.10/site-packages/numpy/__init__.py
pandas: 2.3.3 /root/llm/je/lib/python3.10/site-packages/pandas/__init__.py


In [2]:
from pathlib import Path
import os
import sys
import subprocess
import json
import pandas as pd
from datetime import datetime

import torch

In [3]:
!pwd

/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/notebooks


In [4]:
path = "/root/llm/JOILang-Server"
#path = "/home/mgjeong/Desktop/llm/JOILang-Server"

In [6]:
py_path = "/root/llm/je/bin/python"
#py_path = "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"

In [7]:
# =============================================================================
# 0. Kernel / Python 환경 확인
# =============================================================================
print("=" * 100)
print("0. Kernel / Python 환경 확인")
print("KERNEL PYTHON:", sys.executable)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("torch cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "CUDA is not available in the current Jupyter kernel. "
        f"Kernel이 {py_path}인지 확인하세요."
    )

# resolve() 사용 금지: /root/llm/je/bin/python이 anaconda 원본으로 풀릴 수 있음
# /home/mgjeong/miniconda3/envs/paper-gpu/bin/python
# a100 : "/root/llm/je/bin/python"
EXPECTED_PYTHON = os.path.abspath(py_path)
CURRENT_PYTHON = os.path.abspath(sys.executable)

if CURRENT_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Wrong Jupyter kernel Python.\n"
        f"Expected: {EXPECTED_PYTHON}\n"
        f"Current : {CURRENT_PYTHON}\n"
        f"Jupyter에서 Kernel → Change Kernel → Python (/root/llm/je)로 바꾸세요."
    )


# =============================================================================
# 1. Repository / Script path 설정
# =============================================================================
print("=" * 100)
print("1. Repository / Script path 설정")
REPO = Path(path).absolute()
VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = REPO / "local_models"

assert REPO.exists(), REPO
assert SCRIPT.exists(), SCRIPT

print("REPO:", REPO)
print("VERSION_DIR:", VERSION_DIR)
print("SCRIPT:", SCRIPT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("LOCAL_MODEL_BASE exists:", LOCAL_MODEL_BASE.exists())


# =============================================================================
# 2. JOILang local model / worker 환경변수 설정
# =============================================================================
print("=" * 100)
print("2. JOILang local model / worker 환경변수 설정")
# 이전 실행에서 남아 있을 수 있는 충돌 변수 제거
for key in [
    "JOI_V15_LOCAL_MODEL_NAME",
    "JOI_V14_LOCAL_MODEL_NAME",
    "JOI_V14_WORKER_PYTHON",
    "JOI_V15_PERSISTENT_WORKER",   # 중요: persistent worker를 끄지 않기 위해 제거
]:
    os.environ.pop(key, None)

os.environ["PYTHONUNBUFFERED"] = "1"

# persistent worker는 유지하되, local model 위치만 지정
os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"

# worker도 현재 Jupyter kernel python과 동일하게 고정
os.environ["JOI_V15_WORKER_PYTHON"] = sys.executable

# debug
os.environ["JOI_V15_DEBUG_WORKER"] = "1"
os.environ["JOI_V15_DEBUG_LOG"] = "/tmp/joi_v15_worker_debug.log"

print("JOI_V15_WORKER_PYTHON:", os.environ.get("JOI_V15_WORKER_PYTHON"))
print("JOI_V15_LOCAL_MODEL_BASE_DIR:", os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
print("JOI_V15_LOCAL_DEVICE:", os.environ.get("JOI_V15_LOCAL_DEVICE"))
print("JOI_V15_LOCAL_FILES_ONLY:", os.environ.get("JOI_V15_LOCAL_FILES_ONLY"))
print("JOI_V15_PERSISTENT_WORKER:", os.environ.get("JOI_V15_PERSISTENT_WORKER"))
print("JOI_V15_DEBUG_LOG:", os.environ.get("JOI_V15_DEBUG_LOG"))


# =============================================================================
# 3. subprocess에서도 CUDA가 정상인지 확인
# =============================================================================
print("3. subprocess에서도 CUDA가 정상인지 확인")
subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import sys; "
            "print('subprocess python:', sys.executable); "
            "import torch; "
            "print('subprocess torch:', torch.__version__); "
            "print('subprocess cuda runtime:', torch.version.cuda); "
            "print('subprocess cuda available:', torch.cuda.is_available()); "
            "assert torch.cuda.is_available(), 'CUDA is not available in subprocess'; "
            "print('subprocess gpu:', torch.cuda.get_device_name(0))"
        ),
    ],
    check=True,
)

print("=" * 100)
print("Environment setup complete.")

0. Kernel / Python 환경 확인
KERNEL PYTHON: /root/llm/je/bin/python
pandas: 2.3.3
torch: 2.7.1+cu118
torch cuda runtime: 11.8
cuda available: True
GPU: NVIDIA A100 80GB PCIe
1. Repository / Script path 설정
REPO: /root/llm/JOILang-Server
VERSION_DIR: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413
SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py
RESULTS_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results
LOCAL_MODEL_BASE: /root/llm/JOILang-Server/local_models
LOCAL_MODEL_BASE exists: False
2. JOILang local model / worker 환경변수 설정
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_DEBUG_LOG: /tmp/joi_v15_worker_debug.log
3. subprocess에서도 CUDA가 정상인지 확인
subprocess python: /root/llm/je/bin/python
subprocess torch: 2.7.1+cu118
subprocess cuda runtime: 11.8
subpro

In [28]:
from pathlib import Path
import os
import sys
import subprocess
from datetime import datetime

# 서버별로 여기만 바꾸면 됨
path = "/root/llm/JOILang-Server"
# path = "/home/mgjeong/Desktop/llm/JOILang-Server"

py_path = "/root/llm/je/bin/python"
# py_path = "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"

REPO = Path(path).resolve()
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_ga_search.py"
RESULTS_ROOT = REPO / "gpt_mg/version0_15_update20260413/results"
LOCAL_MODEL_BASE = REPO / "local_models"

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

print("REPO:", REPO)
print("SCRIPT exists:", SCRIPT.exists())
print("RESULTS_ROOT:", RESULTS_ROOT)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE, LOCAL_MODEL_BASE.exists())
print("PYTHON:", py_path, Path(py_path).exists())

REPO: /root/llm/JOILang-Server
SCRIPT exists: True
RESULTS_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results
LOCAL_MODEL_BASE: /root/llm/JOILang-Server/local_models False
PYTHON: /root/llm/je/bin/python True


In [29]:
for model_key, dirname in MODEL_DIRS.items():
    model_path = LOCAL_MODEL_BASE / dirname
    print("\n", model_key)
    print("path:", model_path)
    print("exists:", model_path.exists())
    if model_path.exists():
        print("config:", (model_path / "config.json").exists())
        print("tokenizer_config:", (model_path / "tokenizer_config.json").exists())
        print("tokenizer_json:", (model_path / "tokenizer.json").exists())
        print("safetensors_index:", (model_path / "model.safetensors.index.json").exists())


 qwen25_coder_7b
path: /root/llm/JOILang-Server/local_models/qwen25_coder_7b
exists: False

 llama31_8b
path: /root/llm/JOILang-Server/local_models/llama31_8b
exists: False

 qwen25_coder_14b
path: /root/llm/JOILang-Server/local_models/qwen25_coder_14b
exists: False

 phi35_mini
path: /root/llm/JOILang-Server/local_models/phi35_mini
exists: False

 gemma2_9b_it
path: /root/llm/JOILang-Server/local_models/gemma2_9b_it
exists: False


In [30]:
from pathlib import Path

# 서버별 repo 경로
path = "/root/llm/JOILang-Server"
# path = "/home/mgjeong/Desktop/llm/JOILang-Server"

# 서버별 python 경로
py_path = "/root/llm/je/bin/python"
# py_path = "/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"

# 서버별 local model 경로: 실제 find 결과에 맞게 수정
local_model_base = "/root/llm/local_models"
# local_model_base = "/root/llm/JOILang-Server/local_models"
# local_model_base = "/home/mgjeong/Desktop/llm/local_models"

REPO = Path(path).resolve()
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_ga_search.py"
RESULTS_ROOT = REPO / "gpt_mg/version0_15_update20260413/results"
LOCAL_MODEL_BASE = Path(local_model_base).resolve()

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

print("REPO:", REPO, REPO.exists())
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE, LOCAL_MODEL_BASE.exists())

REPO: /root/llm/JOILang-Server True
SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py True
LOCAL_MODEL_BASE: /root/llm/local_models False


In [31]:
for model_key, dirname in MODEL_DIRS.items():
    model_path = LOCAL_MODEL_BASE / dirname
    print("\n", model_key)
    print("path:", model_path)
    print("exists:", model_path.exists())
    if model_path.exists():
        print("config:", (model_path / "config.json").exists())
        print("tokenizer_config:", (model_path / "tokenizer_config.json").exists())
        print("tokenizer_json:", (model_path / "tokenizer.json").exists())
        print("safetensors_index:", (model_path / "model.safetensors.index.json").exists())


 qwen25_coder_7b
path: /root/llm/local_models/qwen25_coder_7b
exists: False

 llama31_8b
path: /root/llm/local_models/llama31_8b
exists: False

 qwen25_coder_14b
path: /root/llm/local_models/qwen25_coder_14b
exists: False

 phi35_mini
path: /root/llm/local_models/phi35_mini
exists: False

 gemma2_9b_it
path: /root/llm/local_models/gemma2_9b_it
exists: False


In [33]:
from pathlib import Path

local_model_base = "/root/llm/local_models"
LOCAL_MODEL_BASE = Path(local_model_base)

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
}

for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    print("\n", model_key)
    print("path:", p)
    print("exists:", p.exists())
    if p.exists():
        print("config:", (p / "config.json").exists())
        print("tokenizer:", (p / "tokenizer.json").exists())
        print("index:", (p / "model.safetensors.index.json").exists())


 qwen25_coder_7b
path: /root/llm/local_models/qwen25_coder_7b
exists: False

 llama31_8b
path: /root/llm/local_models/llama31_8b
exists: False

 qwen25_coder_14b
path: /root/llm/local_models/qwen25_coder_14b
exists: False


In [8]:
from pathlib import Path
import os
import sys
import subprocess
from datetime import datetime

def run_ga_smoke_pair(
    label: str,
    model_key: str,
    target_detpass: float,
    use_cloud_advisor: bool,
    categories=(1, 2),
    limit_per_category=2,
    population=2,
    gens=3,
    sample_size=4,
    validation_size=4,
    timeout_sec=600,
    advisor_trigger_mode: str = "always",
    advisor_min_population_for_child: int = 4,
    advisor_force_child_quota: bool = True,
    use_mock_advisor: bool = False,
    launcher_python: str | None = None,
    worker_python: str | None = None,
    force_worker_mode: bool = False,
):
    if launcher_python is None:
        launcher_python = sys.executable
    if worker_python is None:
        worker_python = launcher_python

    # resolve() 금지: /root/llm/je/bin/python이 anaconda 원본으로 풀릴 수 있음
    launcher_python = os.path.abspath(os.path.expanduser(launcher_python))
    worker_python = os.path.abspath(os.path.expanduser(worker_python))

    if not Path(launcher_python).exists():
        raise FileNotFoundError(f"launcher_python does not exist: {launcher_python}")
    if not Path(worker_python).exists():
        raise FileNotFoundError(f"worker_python does not exist: {worker_python}")

    if use_cloud_advisor and not use_mock_advisor:
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError(
                "OPENAI_API_KEY is not set. "
                "Set it with getpass before running real cloud-advisor mode."
            )

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    cat_text = "".join(str(c) for c in categories)

    mode = "cloud_advisor" if use_cloud_advisor else "cloudless"
    mock_tag = "_mock" if use_mock_advisor and use_cloud_advisor else ""
    worker_tag = "_forcedworker" if force_worker_mode else "_persistent_default"
    safe_model_key = model_key.replace("/", "_").replace(":", "_")

    out_dir = (
        RESULTS_ROOT
        / f"ga_smoke_{mode}{mock_tag}{worker_tag}_cat{cat_text}_lpc{limit_per_category}"
          f"_pop{population}_gens{gens}_{safe_model_key}_{ts}"
        / "ga_output"
    )

    cmd = [
        launcher_python,
        "-u",
        str(SCRIPT),
        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),

        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),

        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "1",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--feedback-guided-mutation",

        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",

        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",

        "--progress", "verbose",
        "--timeout-sec", str(timeout_sec),
        "--retries", "0",
        "--full-run",
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(out_dir),
    ]

    # 기본은 False. True로 하면 version0_13/qwen_local_worker.py fallback 경로를 탈 수 있음.
    if force_worker_mode:
        cmd += ["--llm-mode", "worker"]

    for c in categories:
        cmd += ["--category", str(c)]

    if use_cloud_advisor:
        cmd += [
            "--llm-mutation-advisor",
            "--advisor-model-key", "gpt41_mini",
            "--advisor-trigger-mode", advisor_trigger_mode,
            "--advisor-min-population-for-child", str(advisor_min_population_for_child),
        ]

        if advisor_force_child_quota:
            cmd += ["--advisor-force-child-quota"]

        if use_mock_advisor:
            cmd += ["--llm-mode", "mock"]
    else:
        cmd += ["--advisor-trigger-mode", "off"]

    run_env = os.environ.copy()

    # A6000 성공 조건과 맞추기 위해 강제 local binding / persistent-off 제거
    # 충돌 변수만 제거
    for k in [
        "JOI_V15_LOCAL_MODEL_NAME",
        "JOI_V14_LOCAL_MODEL_NAME",
        "JOI_V14_WORKER_PYTHON",
        "JOI_V15_PERSISTENT_WORKER",  # 중요: false로 남아 있으면 안 됨
    ]:
        run_env.pop(k, None)

    run_env["PYTHONUNBUFFERED"] = "1"
    run_env["JOI_V15_WORKER_PYTHON"] = worker_python
    
    # local model은 명시
    run_env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(REPO / "local_models")
    run_env["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
    run_env["JOI_V15_LOCAL_FILES_ONLY"] = "true"
    
    # debug log는 run별로 분리
    debug_log = f"/tmp/joi_v15_worker_debug_{safe_model_key}_{ts}.log"
    run_env["JOI_V15_DEBUG_WORKER"] = "1"
    run_env["JOI_V15_DEBUG_LOG"] = debug_log

    print("=" * 100)
    print(f"RUN: {label} / {model_key} / {mode}{mock_tag}{worker_tag}")
    print("OUTPUT:", out_dir)
    print("KERNEL_PYTHON:", sys.executable)
    print("LAUNCHER_PYTHON:", launcher_python)
    print("WORKER_PYTHON:", run_env.get("JOI_V15_WORKER_PYTHON"))
    print("FORCE_WORKER_MODE:", force_worker_mode)
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", run_env.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
    print("JOI_V15_LOCAL_DEVICE:", run_env.get("JOI_V15_LOCAL_DEVICE"))
    print("JOI_V15_LOCAL_FILES_ONLY:", run_env.get("JOI_V15_LOCAL_FILES_ONLY"))
    print("JOI_V15_PERSISTENT_WORKER:", run_env.get("JOI_V15_PERSISTENT_WORKER"))
    print("JOI_V15_LOCAL_MODEL_NAME:", run_env.get("JOI_V15_LOCAL_MODEL_NAME"))
    print("DEBUG_LOG:", debug_log)
    print("COMMAND:")
    print(" ".join(cmd))
    print("=" * 100)

    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()

    print("\nRETURN CODE:", rc)
    print("OUTPUT:", out_dir)
    print("DEBUG_LOG:", debug_log)

    if rc != 0:
        raise RuntimeError(
            f"{label} {mode}{mock_tag}{worker_tag} failed with return code {rc}"
        )

    return out_dir, debug_log

In [20]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_worker_cat1_sample1",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1,),
    limit_per_category=1,
    population=1,
    gens=1,
    sample_size=1,
    validation_size=1,
    timeout_sec=600,
    launcher_python=py_path,
    worker_python=py_path,
    force_worker_mode=True,
)

RUN: smoke_worker_cat1_sample1 / qwen25_coder_7b / cloudless_forcedworker
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_forcedworker_cat1_lpc1_pop1_gens1_qwen25_coder_7b_20260602_221916/ga_output
KERNEL_PYTHON: /root/llm/je/bin/python
LAUNCHER_PYTHON: /root/llm/je/bin/python
WORKER_PYTHON: /root/llm/je/bin/python
FORCE_WORKER_MODE: True
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_LOCAL_MODEL_NAME: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_221916.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 100.0 --population 1 --gens 1 --min-generations 1 --max-generations 1 --sample-size 1 --validation-size 1 --cheap-eval-limit 1 --candidate-k 1 --repair-attempts 0

#### debug log를 확인

In [25]:
from pathlib import Path
import json
import pandas as pd

def _safe_read_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"[JSON READ ERROR] {path}: {type(e).__name__}: {e}")
        return None

def inspect_worker_crash_detail(ga_out, debug_log=None, max_csv=6, max_logs_per_csv=2):
    ga_out = Path(ga_out)

    print("=" * 120)
    print("GA_OUT:", ga_out)
    print("=" * 120)

    if not ga_out.exists():
        print("[ERROR] ga_out does not exist:", ga_out)
        return

    # 1) summary/progress quick check
    summary_path = ga_out / "ga_summary.json"
    if summary_path.exists():
        print("\n[ga_summary.json]")
        s = _safe_read_json(summary_path) or {}
        for k in [
            "model_key",
            "best_DETPass",
            "best_so_far_DETPass",
            "accepted_best_DETPass",
            "best_avg_DET",
            "stop_reason",
            "final_phase",
            "advisor_used",
            "cloudless_mutation_used",
        ]:
            if k in s:
                print(f"{k}: {s.get(k)}")

    progress_path = ga_out / "ga_generation_progress.csv"
    if progress_path.exists():
        print("\n[ga_generation_progress.csv]")
        try:
            pdf = pd.read_csv(progress_path)
            cols = [
                "generation",
                "model_key",
                "validation_det_pass_rate",
                "validation_avg_det_score",
                "best_so_far_DETPass",
                "avg_prompt_tokens",
                "genome_id",
            ]
            cols = [c for c in cols if c in pdf.columns]
            display(pdf[cols])
        except Exception as e:
            print("[progress read error]", type(e).__name__, e)

    # 2) candidate csv inspection
    cand_dir = ga_out / "candidates"
    cand_files = sorted(cand_dir.glob("*.csv"))

    print("\n[candidates]")
    print("candidate dir:", cand_dir)
    print("candidate csv count:", len(cand_files))

    if not cand_files:
        print("[ERROR] no candidate CSV files found")
    else:
        for csv_path in cand_files[:max_csv]:
            print("\n" + "-" * 120)
            print("CSV:", csv_path.name)

            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print("[CSV READ ERROR]", type(e).__name__, e)
                continue

            cols = [
                "llm_mode",
                "llm_model",
                "generation_status",
                "generation_error_count",
                "generation_error_type",
                "generation_error_types",
                "generation_prompt_tokens_total",
                "generation_completion_tokens_total",
                "generation_total_tokens_total",
                "generation_peak_vram_gb",
                "candidate_count",
                "candidates",
                "prompt_log_paths",
            ]
            cols = [c for c in cols if c in df.columns]
            display(df[cols].head(10))

            if "generation_status" in df.columns:
                print("generation_status counts:")
                print(df["generation_status"].value_counts(dropna=False).to_string())

            if "generation_error_type" in df.columns:
                print("generation_error_type counts:")
                print(df["generation_error_type"].value_counts(dropna=False).to_string())

            if "candidates" in df.columns:
                print("\n[first candidates]")
                for x in df["candidates"].head(3).tolist():
                    print(str(x)[:1500])
                    print("-" * 80)

            # 3) prompt logs
            if "prompt_log_paths" not in df.columns:
                continue

            rows_with_logs = df["prompt_log_paths"].dropna().head(max_logs_per_csv).tolist()
            for raw_paths in rows_with_logs:
                try:
                    paths = json.loads(raw_paths)
                except Exception:
                    print("[prompt_log_paths parse failed]", str(raw_paths)[:1000])
                    continue

                for log_path in paths[:max_logs_per_csv]:
                    p = Path(log_path)
                    print("\n[PROMPT LOG]", p)
                    print("exists:", p.exists())

                    if not p.exists():
                        continue

                    obj = _safe_read_json(p)
                    if not obj:
                        continue

                    req = obj.get("request") or {}
                    resp = obj.get("response") or {}

                    print("\n[request]")
                    for k in [
                        "mode",
                        "model",
                        "worker_python",
                        "worker_path",
                        "worker_runtime",
                        "resolved_local_model_name",
                        "local_device",
                        "local_dtype",
                        "local_files_only",
                        "local_load_in_4bit",
                        "local_attn_implementation",
                        "local_hf_modules_cache",
                        "endpoint",
                    ]:
                        print(f"{k}: {req.get(k)}")

                    print("\n[top-level error]")
                    print(str(obj.get("error"))[:4000])

                    print("\n[response]")
                    for k in [
                        "content",
                        "prompt_tokens",
                        "completion_tokens",
                        "total_tokens",
                        "peak_vram_gb",
                        "latency_sec",
                        "attempt",
                    ]:
                        v = resp.get(k)
                        print(f"{k}: {str(v)[:2000]}")

                    print("\n[raw]")
                    raw = resp.get("raw")
                    if isinstance(raw, dict):
                        for k, v in raw.items():
                            print(f"{k}: {str(v)[:4000]}")
                    else:
                        print(str(raw)[:4000])

    # 4) debug log tail
    if debug_log is not None:
        p = Path(debug_log)
        print("\n" + "=" * 120)
        print("[DEBUG LOG]")
        print("debug_log:", p)
        print("exists:", p.exists())

        if p.exists():
            txt = p.read_text(errors="replace")
            print(txt[-12000:])
        else:
            print("[WARN] debug log not found")

In [26]:
inspect_worker_crash_detail(out, debug_log)

GA_OUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_forcedworker_cat1_lpc1_pop1_gens1_qwen25_coder_7b_20260602_221916/ga_output

[ga_summary.json]
best_DETPass: 0.0
best_so_far_DETPass: 0.0
accepted_best_DETPass: 0.0
best_avg_DET: 0.0
stop_reason: max generations reached
final_phase: FINAL_SELECTION
advisor_used: False
cloudless_mutation_used: True

[ga_generation_progress.csv]


,generation,model_key,validation_det_pass_rate,validation_avg_det_score,best_so_far_DETPass,avg_prompt_tokens,genome_id
0,1,qwen25_coder_7b,0.0,0.0,0.0,0.0,gen-378892e9-ecc3-87ab-8b45-85023a0286cc



[candidates]
candidate dir: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_forcedworker_cat1_lpc1_pop1_gens1_qwen25_coder_7b_20260602_221916/ga_output/candidates
candidate csv count: 2

------------------------------------------------------------------------------------------------------------------------
CSV: candidates_ga_quick_Qwen_Qwen2.5-Coder-7B-Instruct_gen-378892e9-ecc3-87ab-8b45-85023a0286cc_7846.csv


,llm_mode,llm_model,generation_status,generation_error_count,generation_error_type,generation_error_types,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,generation_peak_vram_gb,candidate_count,candidates,prompt_log_paths
0,worker,Qwen/Qwen2.5-Coder-7B-Instruct,partial_error,1,worker_crash,"[""worker_crash""]",0,0,0,0.0,1,"[""""]","[""/root/llm/JOILang-Server/gpt_mg/version0_15_..."


generation_status counts:
generation_status
partial_error    1
generation_error_type counts:
generation_error_type
worker_crash    1

[first candidates]
[""]
--------------------------------------------------------------------------------

[PROMPT LOG] /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/logs/ga_quick_Qwen_Qwen2.5-Coder-7B-Instruct_gen-378892e9-ecc3-87ab-8b45-85023a0286cc_7846/row_001_cand_1.json
exists: True

[request]
mode: None
model: None
worker_python: None
worker_path: None
worker_runtime: None
resolved_local_model_name: None
local_device: None
local_dtype: None
local_files_only: None
local_load_in_4bit: None
local_attn_implementation: None
local_hf_modules_cache: None
endpoint: None

[top-level error]
We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'

,llm_mode,llm_model,generation_status,generation_error_count,generation_error_type,generation_error_types,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,generation_peak_vram_gb,candidate_count,candidates,prompt_log_paths
0,worker,Qwen/Qwen2.5-Coder-7B-Instruct,partial_error,1,worker_crash,"[""worker_crash""]",0,0,0,0.0,1,"[""""]","[""/root/llm/JOILang-Server/gpt_mg/version0_15_..."


generation_status counts:
generation_status
partial_error    1
generation_error_type counts:
generation_error_type
worker_crash    1

[first candidates]
[""]
--------------------------------------------------------------------------------

[PROMPT LOG] /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/logs/ga_validation_Qwen_Qwen2.5-Coder-7B-Instruct_gen-378892e9-ecc3-87ab-8b45-85023a0286cc_507846/row_001_cand_1.json
exists: True

[request]
mode: None
model: None
worker_python: None
worker_path: None
worker_runtime: None
resolved_local_model_name: None
local_device: None
local_dtype: None
local_files_only: None
local_load_in_4bit: None
local_attn_implementation: None
local_hf_modules_cache: None
endpoint: None

[top-level error]
We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offlin

In [22]:
from pathlib import Path

p = Path(debug_log)
print("debug_log:", p)
print("exists:", p.exists())

if p.exists():
    text = p.read_text(encoding="utf-8", errors="replace")
    print(text[-10000:])

debug_log: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_221916.log
exists: True
offline-mode'.\n\nDuring handling of the above exception, another exception occurred:\n\nTraceback (most recent call last):\n  File \"/root/llm/je/lib/python3.10/site-packages/transformers/utils/hub.py\", line 422, in cached_files\n    hf_hub_download(\n  File \"/root/llm/je/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py\", line 88, in _inner_fn\n    return fn(*args, **kwargs)\n  File \"/root/llm/je/lib/python3.10/site-packages/huggingface_hub/file_download.py\", line 997, in hf_hub_download\n    return _hf_hub_download_to_cache_dir(\n  File \"/root/llm/je/lib/python3.10/site-packages/huggingface_hub/file_download.py\", line 1148, in _hf_hub_download_to_cache_dir\n    _raise_on_head_call_error(head_call_error, force_download, local_files_only)\n  File \"/root/llm/je/lib/python3.10/site-packages/huggingface_hub/file_download.py\", line 1773, in _raise_on_head_call_error\n    raise Lo

In [11]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat1_sample2",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1,),
    limit_per_category=2,
    population=2,
    gens=1,
    sample_size=2,
    validation_size=2,
    timeout_sec=600,
    launcher_python=py_path,
    worker_python=py_path,
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat1_sample2 / qwen25_coder_7b / cloudless_persistent_default
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_persistent_default_cat1_lpc2_pop2_gens1_qwen25_coder_7b_20260602_221209/ga_output
KERNEL_PYTHON: /root/llm/je/bin/python
LAUNCHER_PYTHON: /root/llm/je/bin/python
WORKER_PYTHON: /root/llm/je/bin/python
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_LOCAL_MODEL_NAME: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_221209.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 100.0 --population 2 --gens 1 --min-generations 1 --max-generations 1 --sample-size 2 --validation-size 2 --cheap-eval-limit 1 --candidate-k 1 --r

## 2단계: category 1,2 smoke

In [12]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat12",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1, 2),
    limit_per_category=2,
    population=2,
    gens=2,
    sample_size=4,
    validation_size=4,
    timeout_sec=900,
    launcher_python=py_path,
    worker_python=py_path,
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat12 / qwen25_coder_7b / cloudless_persistent_default
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_persistent_default_cat12_lpc2_pop2_gens2_qwen25_coder_7b_20260602_221224/ga_output
KERNEL_PYTHON: /root/llm/je/bin/python
LAUNCHER_PYTHON: /root/llm/je/bin/python
WORKER_PYTHON: /root/llm/je/bin/python
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_LOCAL_MODEL_NAME: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_221224.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 100.0 --population 2 --gens 2 --min-generations 2 --max-generations 2 --sample-size 4 --validation-size 4 --cheap-eval-limit 1 --candidate-k 1 --repair-

## 3단계: 기존 GA 설정에 가깝게 확대

In [13]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat12_pop5_gens3",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1, 2),
    limit_per_category=3,
    population=5,
    gens=3,
    sample_size=6,
    validation_size=6,
    timeout_sec=1800,
    launcher_python=py_path,
    worker_python=py_path,
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat12_pop5_gens3 / qwen25_coder_7b / cloudless_persistent_default
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_persistent_default_cat12_lpc3_pop5_gens3_qwen25_coder_7b_20260602_221307/ga_output
KERNEL_PYTHON: /root/llm/je/bin/python
LAUNCHER_PYTHON: /root/llm/je/bin/python
WORKER_PYTHON: /root/llm/je/bin/python
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_LOCAL_MODEL_NAME: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_221307.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 100.0 --population 5 --gens 3 --min-generations 3 --max-generations 3 --sample-size 6 --validation-size 6 --cheap-eval-limit 1 --candidate-k 

# 3. 본격 테스트
## “환경 설정 → 실행 wrapper → 6개 run 실행 → 요약/Delta 생성”
A6000 SetA = 중간 규모, 안정성/비용/시간 균형

A100  SetB = 원래 full cloudless 조건에 가까운 본 실험 규모



## API key 입력 셀

In [38]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## Cell 1. 기본 경로 / 환경 / 서버 preset 설정


In [39]:

from pathlib import Path
import os
import sys
import json
import time
import subprocess
import traceback
from datetime import datetime

import pandas as pd


# =============================================================================
# 0. Server preset 선택
# =============================================================================

# A6000 서버에서는 이 값 사용
# SERVER_PRESET = "A6000_SET_A"

# A100 서버에서는 위 줄을 주석 처리하고 아래 줄 사용
SERVER_PRESET = "A100_SET_B"
path = "/root/llm/JOILang-Server"
py_path = "/root/llm/je/bin/python"
local_model_base = "/root/llm/local_models"
# =============================================================================
# 1. Repository / script path 설정
# =============================================================================

try:
    REPO = Path(path).absolute()
except NameError:
    REPO = Path.cwd().absolute()

VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = Path(local_model_base).absolute()


assert REPO.exists(), REPO
assert SCRIPT.exists(), SCRIPT

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("SERVER_PRESET:", SERVER_PRESET)
print("REPO:", REPO)
print("SCRIPT:", SCRIPT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("LOCAL_MODEL_BASE exists:", LOCAL_MODEL_BASE.exists())
print("PYTHON:", sys.executable)


# =============================================================================
# 2. 모델 목록
# =============================================================================

MODEL_LIST = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]

RUN_MODES = [
    ("cloudless", False),
    ("cloud_advisor", True),
]


# =============================================================================
# 3. 서버별 실험 설정
# =============================================================================

# A6000: 중간 규모. 48GB급에서 14B까지 안정적으로 비교하기 위한 SetA.
SET_A_A6000 = dict(
    target_detpass=90,
    categories=range(1, 9),
    limit_per_category=2,
    sample_size=16,
    validation_size=16,
    population=4,
    gens=6,
    full_run=True,
    progress="verbose",
    timeout_sec=900,
    retries=0,
    idle_timeout_sec=2400,
    total_timeout_sec=24 * 3600,
)

# A100: 기존 cloudless full 설정에 가까운 본 비교 SetB.
SET_B_A100 = dict(
    target_detpass=90,
    categories=range(1, 9),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    full_run=True,
    progress="verbose",
    timeout_sec=1200,
    retries=0,
    idle_timeout_sec=3600,
    total_timeout_sec=40 * 3600,
)

if SERVER_PRESET == "A6000_SET_A":
    COMMON_GA_CONFIG = SET_A_A6000
elif SERVER_PRESET == "A100_SET_B":
    COMMON_GA_CONFIG = SET_B_A100
else:
    raise ValueError(f"Unknown SERVER_PRESET: {SERVER_PRESET}")

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
COMPARISON_ROOT = RESULTS_ROOT / f"fair_compare_{SERVER_PRESET}_{RUN_TAG}"
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("COMMON_GA_CONFIG:", COMMON_GA_CONFIG)
print("COMPARISON_ROOT:", COMPARISON_ROOT)


# =============================================================================
# 4. 환경변수 설정
# =============================================================================

def setup_env_for_current_server():
    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ["JOI_V15_WORKER_PYTHON"] = sys.executable

    # 공통 충돌 변수 제거
    os.environ.pop("JOI_V15_LOCAL_MODEL_NAME", None)
    os.environ.pop("JOI_V14_LOCAL_MODEL_NAME", None)
    os.environ.pop("JOI_V14_WORKER_PYTHON", None)

    # 중요: persistent worker를 끄면 안 됨.
    # false가 남아 있으면 version0_13/qwen_local_worker.py fallback 가능.
    os.environ.pop("JOI_V15_PERSISTENT_WORKER", None)

    if SERVER_PRESET == "A100_SET_B":
        # A100에서는 offline local model 경로를 명확히 고정
        os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
        os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
        os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"

    elif SERVER_PRESET == "A6000_SET_A":
        # A6000에서 기존 성공 환경을 최대한 보존.
        # 이미 잘 도는 경우 LOCAL_MODEL_BASE_DIR을 강제하지 않음.
        # 필요할 때만 아래 3줄을 켜면 됨.
        # os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
        # os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
        # os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
        pass

    print("=" * 100)
    print("JOI_V15_WORKER_PYTHON:", os.environ.get("JOI_V15_WORKER_PYTHON"))
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
    print("JOI_V15_LOCAL_DEVICE:", os.environ.get("JOI_V15_LOCAL_DEVICE"))
    print("JOI_V15_LOCAL_FILES_ONLY:", os.environ.get("JOI_V15_LOCAL_FILES_ONLY"))
    print("JOI_V15_PERSISTENT_WORKER:", os.environ.get("JOI_V15_PERSISTENT_WORKER"))

setup_env_for_current_server()


# =============================================================================
# 5. Cloud advisor API key 확인
# =============================================================================

if not os.environ.get("OPENAI_API_KEY"):
    print("[WARN] OPENAI_API_KEY is not set. cloud_advisor runs will fail unless set.")
else:
    print("[OK] OPENAI_API_KEY is set.")

SERVER_PRESET: A100_SET_B
REPO: /root/llm/JOILang-Server
SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py
RESULTS_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results
LOCAL_MODEL_BASE: /root/llm/local_models
LOCAL_MODEL_BASE exists: True
PYTHON: /root/llm/je/bin/python
COMMON_GA_CONFIG: {'target_detpass': 90, 'categories': range(1, 9), 'limit_per_category': 3, 'sample_size': 24, 'validation_size': 24, 'population': 5, 'gens': 10, 'full_run': True, 'progress': 'verbose', 'timeout_sec': 1200, 'retries': 0, 'idle_timeout_sec': 3600, 'total_timeout_sec': 144000}
COMPARISON_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/fair_compare_A100_SET_B_20260609_203929
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
[OK] OPENAI_API_KEY is set.


## Cell 2. run_ga_all_categories wrapper 정의


In [40]:
# =============================================================================
# run_ga_search.py CLI wrapper
# =============================================================================

def _get_run_ga_help_text():
    try:
        proc = subprocess.run(
            [sys.executable, str(SCRIPT), "--help"],
            cwd=str(REPO),
            env=os.environ.copy(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=60,
        )
        return proc.stdout or ""
    except Exception as e:
        print("[WARN] failed to inspect --help:", repr(e))
        return ""

_RUN_GA_HELP_TEXT = _get_run_ga_help_text()

def _has_flag(flag: str) -> bool:
    if not _RUN_GA_HELP_TEXT:
        return True
    return flag in _RUN_GA_HELP_TEXT

def _append_optional_flag(cmd, flag, value=None):
    if _has_flag(flag):
        cmd.append(flag)
        if value is not None:
            cmd.append(str(value))
    else:
        print(f"[SKIP unsupported flag] {flag}")


def run_ga_all_categories(
    model_key: str,
    categories=range(1, 9),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    target_detpass=90,
    base_prefix="ga_final",
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=600,
    retries=0,
    idle_timeout_sec=2400,
    total_timeout_sec=32 * 3600,

    advisor_trigger_mode="always",
    advisor_min_population_for_child=4,
    advisor_force_child_quota=True,
    use_mock_advisor=False,

    launcher_python=None,
):
    if launcher_python is None:
        launcher_python = py_path

    launcher_python = os.path.abspath(os.path.expanduser(launcher_python))

    if not Path(launcher_python).exists():
        raise FileNotFoundError(f"launcher_python does not exist: {launcher_python}")

    if model_key not in MODEL_DIRS:
        raise KeyError(f"Unknown model_key={model_key}. Available={list(MODEL_DIRS)}")

    local_model_path = (LOCAL_MODEL_BASE / MODEL_DIRS[model_key]).absolute()

    if not local_model_path.exists():
        raise FileNotFoundError(f"local model path does not exist: {local_model_path}")
    if not (local_model_path / "config.json").exists():
        raise FileNotFoundError(f"config.json not found: {local_model_path}")
    if not (local_model_path / "tokenizer.json").exists():
        raise FileNotFoundError(f"tokenizer.json not found: {local_model_path}")

    if use_advisor and not use_mock_advisor:
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY is not set for real cloud advisor mode.")

    categories = tuple(categories)
    cat_text = "".join(str(c) for c in categories)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    mode = "cloud_advisor" if use_advisor else "cloudless"
    mock_tag = "_mock" if use_advisor and use_mock_advisor else ""
    safe_model_key = model_key.replace("/", "_").replace(":", "_")

    run_name = (
        f"{base_prefix}_{mode}{mock_tag}_cat{cat_text}"
        f"_lpc{limit_per_category}_pop{population}_gens{gens}"
        f"_{safe_model_key}_{ts}"
    )

    out_dir = RESULTS_ROOT / run_name / "ga_output"

    cmd = [
        launcher_python,
        "-u",
        str(SCRIPT),

        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),

        "--llm-mode", "worker",

        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),

        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "2",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--feedback-guided-mutation",

        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",

        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",

        "--progress", str(progress),
        "--timeout-sec", str(timeout_sec),
        "--retries", str(retries),
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(out_dir),
    ]

    _append_optional_flag(cmd, "--idle-timeout-sec", idle_timeout_sec)
    _append_optional_flag(cmd, "--total-timeout-sec", total_timeout_sec)

    if full_run:
        _append_optional_flag(cmd, "--full-run")

    for c in categories:
        cmd += ["--category", str(c)]

    if use_advisor:
        cmd += [
            "--llm-mutation-advisor",
            "--advisor-model-key", "gpt41_mini",
            "--advisor-trigger-mode", advisor_trigger_mode,
            "--advisor-min-population-for-child", str(advisor_min_population_for_child),
        ]

        if advisor_force_child_quota:
            cmd += ["--advisor-force-child-quota"]

        if use_mock_advisor:
            cmd += ["--llm-mode", "mock"]
    else:
        cmd += ["--advisor-trigger-mode", "off"]

    run_env = os.environ.copy()
    run_env["PYTHONUNBUFFERED"] = "1"

    run_env.pop("JOI_V14_WORKER_PYTHON", None)

    run_env["JOI_V15_WORKER_PYTHON"] = launcher_python
    run_env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
    run_env["JOI_V15_LOCAL_MODEL_NAME"] = str(local_model_path)
    run_env["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
    run_env["JOI_V15_LOCAL_FILES_ONLY"] = "true"

    run_env["TRANSFORMERS_OFFLINE"] = "1"
    run_env["HF_HUB_OFFLINE"] = "1"

    run_env["USE_TORCH"] = "1"
    run_env["USE_TF"] = "0"
    run_env["USE_FLAX"] = "0"
    run_env.pop("TRANSFORMERS_NO_TORCH", None)

    if run_env.get("JOI_V15_PERSISTENT_WORKER") == "false":
        run_env.pop("JOI_V15_PERSISTENT_WORKER", None)

    debug_log = f"/tmp/joi_v15_worker_debug_{safe_model_key}_{mode}_{ts}.log"
    run_env["JOI_V15_DEBUG_WORKER"] = "1"
    run_env["JOI_V15_DEBUG_LOG"] = debug_log

    print("=" * 120)
    print(f"RUN: {model_key} / {mode}{mock_tag}")
    print("OUTPUT:", out_dir)
    print("LAUNCHER_PYTHON:", launcher_python)
    print("JOI_V15_WORKER_PYTHON:", run_env.get("JOI_V15_WORKER_PYTHON"))
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", run_env.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
    print("JOI_V15_LOCAL_MODEL_NAME:", run_env.get("JOI_V15_LOCAL_MODEL_NAME"))
    print("JOI_V15_LOCAL_DEVICE:", run_env.get("JOI_V15_LOCAL_DEVICE"))
    print("JOI_V15_LOCAL_FILES_ONLY:", run_env.get("JOI_V15_LOCAL_FILES_ONLY"))
    print("JOI_V15_PERSISTENT_WORKER:", run_env.get("JOI_V15_PERSISTENT_WORKER"))
    print("DEBUG_LOG:", debug_log)
    print("COMMAND:")
    print(" ".join(cmd))
    print("=" * 120)

    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert proc.stdout is not None

    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()

    print("\nRETURN CODE:", rc)
    print("OUTPUT:", out_dir)
    print("DEBUG_LOG:", debug_log)

    if rc != 0:
        raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")

    return out_dir

In [45]:
# ============================================================
# ONE-CELL PRECHECK BEFORE FULL FAIR GA LOOP
# ============================================================

from pathlib import Path
import os
import re
import json
import time
import getpass
import traceback
import subprocess
import pandas as pd

# ============================================================
# 0. OPENAI API KEY CHECK FIRST
# ============================================================

REQUIRE_OPENAI_API_KEY = True  # cloud_advisor까지 테스트할 것이므로 True

def ensure_openai_api_key(required: bool = True):
    key = os.environ.get("OPENAI_API_KEY", "").strip()

    if key:
        print("[OK] OPENAI_API_KEY is already set.")
        return True

    if not required:
        print("[WARN] OPENAI_API_KEY is not set. Cloud advisor test will be skipped.")
        return False

    print("[INPUT REQUIRED] OPENAI_API_KEY is not set.")
    print("Enter your OpenAI API key. It will be stored only in this notebook process environment.")
    key = getpass.getpass("OPENAI_API_KEY: ").strip()

    if not key:
        raise RuntimeError("OPENAI_API_KEY was not provided.")

    if not key.startswith("sk-"):
        print("[WARN] The key does not start with 'sk-'. Continue only if this is intentional.")

    os.environ["OPENAI_API_KEY"] = key
    print("[OK] OPENAI_API_KEY has been set for this notebook process.")
    return True

OPENAI_KEY_READY = ensure_openai_api_key(required=REQUIRE_OPENAI_API_KEY)


# ============================================================
# 1. SERVER CONFIG
# ============================================================

SERVER_PRESET = "A100_SET_B"

path = "/root/llm/JOILang-Server"
py_path = "/root/llm/je/bin/python"
local_model_base = "/root/llm/local_models"

REPO = Path(path).resolve()
VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = Path(local_model_base).resolve()

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

print("\n" + "=" * 120)
print("[CONFIG]")
print("REPO:", REPO, REPO.exists())
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("PYTHON:", py_path, Path(py_path).exists())
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE, LOCAL_MODEL_BASE.exists())
print("=" * 120)

assert REPO.exists(), REPO
assert SCRIPT.exists(), SCRIPT
assert Path(py_path).exists(), py_path
assert LOCAL_MODEL_BASE.exists(), LOCAL_MODEL_BASE

if "run_ga_all_categories" not in globals():
    raise RuntimeError("run_ga_all_categories is not defined. Define/fix run_ga_all_categories first, then rerun this cell.")


# ============================================================
# 2. PATCH run_ga_search.py: feedback=None guard
# ============================================================

print("\n" + "=" * 120)
print("[PATCH CHECK] feedback=None guard")
print("=" * 120)

src = SCRIPT.read_text(encoding="utf-8")

if "feedback = feedback or {}\n" in src:
    print("[OK] feedback=None guard already exists.")
else:
    new_src, n = re.subn(
        r'^(\s*)if feedback\.get\("improved"\):',
        r'\1feedback = feedback or {}\n\1if feedback.get("improved"):',
        src,
        count=1,
        flags=re.M,
    )
    if n != 1:
        raise RuntimeError(f"Could not patch feedback guard. matched={n}")
    SCRIPT.write_text(new_src, encoding="utf-8")
    print("[PATCHED] Added: feedback = feedback or {}")

subprocess.check_call([py_path, "-m", "py_compile", str(SCRIPT)])
print("[OK] py_compile passed.")


# ============================================================
# 3. MODEL PATH CHECK
# ============================================================

print("\n" + "=" * 120)
print("[MODEL PATH CHECK]")
print("=" * 120)

# MODEL_LIST가 이미 있으면 거기에 맞춰 검사/스모크하고, 없으면 기본 3개만 검사
if "MODEL_LIST" in globals():
    SMOKE_MODEL_LIST = list(MODEL_LIST)
else:
    SMOKE_MODEL_LIST = [
        ("7B", "qwen25_coder_7b"),
        ("8B", "llama31_8b"),
        ("14B", "qwen25_coder_14b"),
    ]

model_rows = []
for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    row = {
        "model_key": model_key,
        "path": str(p),
        "exists": p.exists(),
        "config": (p / "config.json").exists(),
        "tokenizer": (p / "tokenizer.json").exists(),
        "index": (p / "model.safetensors.index.json").exists(),
        "safetensors_count": len(list(p.glob("*.safetensors"))) if p.exists() else 0,
        "size": None,
    }
    try:
        if p.exists():
            row["size"] = subprocess.check_output(["du", "-sh", str(p)], text=True).split()[0]
    except Exception:
        pass
    model_rows.append(row)

model_df = pd.DataFrame(model_rows)
display(model_df)

required_model_keys = [mk for _, mk in SMOKE_MODEL_LIST]
for mk in required_model_keys:
    if mk not in MODEL_DIRS:
        raise KeyError(f"Unknown model key in MODEL_LIST: {mk}")

    p = LOCAL_MODEL_BASE / MODEL_DIRS[mk]
    assert p.exists(), f"Missing model dir: {p}"
    assert (p / "config.json").exists(), f"Missing config.json: {p}"
    assert (p / "tokenizer.json").exists(), f"Missing tokenizer.json: {p}"
    assert (p / "model.safetensors.index.json").exists(), f"Missing model.safetensors.index.json: {p}"
    assert len(list(p.glob("*.safetensors"))) > 0, f"No safetensors files: {p}"

print("[OK] Required smoke models are available:", required_model_keys)


# ============================================================
# 4. HELPERS
# ============================================================

def summarize_ga_output(name, out_dir):
    out_dir = Path(out_dir)
    summary_path = out_dir / "ga_summary.json"
    progress_path = out_dir / "ga_generation_progress.csv"
    advisor_path = out_dir / "advisor_mutation_proposals.jsonl"
    proposal_path = out_dir / "mutation_proposals.jsonl"

    result = {
        "run": name,
        "ok": False,
        "out_dir": str(out_dir),
        "summary_exists": summary_path.exists(),
        "progress_exists": progress_path.exists(),
        "best_DETPass": None,
        "best_so_far_DETPass": None,
        "accepted_best_DETPass": None,
        "validation_det_pass_rate": None,
        "validation_avg_det_score": None,
        "avg_prompt_tokens": None,
        "tokens_gt_0": False,
        "advisor_used": None,
        "cloudless_mutation_used": None,
        "advisor_proposal_count": 0,
        "proposal_sources": "",
        "stop_reason": None,
        "final_phase": None,
    }

    if summary_path.exists():
        try:
            s = json.loads(summary_path.read_text(encoding="utf-8"))
            for k in [
                "best_DETPass",
                "best_so_far_DETPass",
                "accepted_best_DETPass",
                "advisor_used",
                "cloudless_mutation_used",
                "stop_reason",
                "final_phase",
            ]:
                result[k] = s.get(k)
        except Exception as e:
            result["summary_error"] = repr(e)

    if progress_path.exists():
        try:
            df = pd.read_csv(progress_path)
            if len(df) > 0:
                last = df.iloc[-1]
                for k in [
                    "validation_det_pass_rate",
                    "validation_avg_det_score",
                    "avg_prompt_tokens",
                    "best_so_far_DETPass",
                    "accepted_best_DETPass",
                    "advisor_used",
                    "cloudless_mutation_used",
                    "generation_phase",
                    "next_action",
                    "plateau_type",
                ]:
                    if k in df.columns:
                        result[k] = last.get(k)

                if "avg_prompt_tokens" in df.columns:
                    result["tokens_gt_0"] = bool(
                        pd.to_numeric(df["avg_prompt_tokens"], errors="coerce").fillna(0).max() > 0
                    )
        except Exception as e:
            result["progress_error"] = repr(e)

    if advisor_path.exists():
        lines = [x for x in advisor_path.read_text(encoding="utf-8").splitlines() if x.strip()]
        result["advisor_proposal_count"] = len(lines)

    if proposal_path.exists():
        sources = []
        for line in proposal_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                sources.append(json.loads(line).get("source"))
            except Exception:
                pass
        result["proposal_sources"] = ",".join(sorted(set(str(x) for x in sources if x)))

    result["ok"] = bool(result["summary_exists"] and result["progress_exists"] and result["tokens_gt_0"])
    return result


def print_candidate_error_tail(out_dir, max_files=3):
    out_dir = Path(out_dir)
    cand_dir = out_dir / "candidates"
    if not cand_dir.exists():
        print("[NO CANDIDATES DIR]", cand_dir)
        return

    csvs = sorted(cand_dir.glob("*.csv"))
    print(f"[CANDIDATE CSV COUNT] {len(csvs)}")

    for csv_path in csvs[:max_files]:
        print("-" * 120)
        print("CSV:", csv_path.name)
        try:
            df = pd.read_csv(csv_path)
            cols = [
                "llm_mode",
                "llm_model",
                "generation_status",
                "generation_error_count",
                "generation_error_type",
                "generation_error_types",
                "generation_prompt_tokens_total",
                "generation_total_tokens_total",
                "candidate_count",
                "candidates",
                "prompt_log_paths",
            ]
            cols = [c for c in cols if c in df.columns]
            display(df[cols])
        except Exception as e:
            print("read error:", repr(e))


# ============================================================
# 5. CLOUDLESS SMOKE: all MODEL_LIST models
# ============================================================

print("\n" + "=" * 120)
print("[SMOKE RUNS] cloudless")
print("=" * 120)

smoke_results = {}
smoke_rows = []

for label, model_key in SMOKE_MODEL_LIST:
    run_name = f"{label}_cloudless_smoke"

    print("\n" + "#" * 120)
    print(f"START {run_name}: {model_key}")
    print("#" * 120)

    try:
        out = run_ga_all_categories(
            model_key=model_key,
            categories=(1,),
            limit_per_category=1,
            sample_size=1,
            validation_size=1,
            population=1,
            gens=1,
            target_detpass=90,
            base_prefix=f"smoke_pathcheck_{SERVER_PRESET}",
            use_advisor=False,
            full_run=True,
            progress="verbose",
            timeout_sec=1200,
            retries=0,
        )

        smoke_results[run_name] = out
        row = summarize_ga_output(run_name, out)
        smoke_rows.append(row)

        print(f"[PASS] {run_name}: {out}")
        print("[SUMMARY]", row)

    except Exception as e:
        print(f"[FAIL] {run_name}: {type(e).__name__}: {e}")
        traceback.print_exc()
        smoke_results[run_name] = None
        smoke_rows.append({
            "run": run_name,
            "ok": False,
            "out_dir": None,
            "error": repr(e),
        })

    time.sleep(3)


# ============================================================
# 6. REAL CLOUD ADVISOR SMOKE: 14B small run
# ============================================================

print("\n" + "=" * 120)
print("[SMOKE RUN] real cloud advisor 14B")
print("=" * 120)

advisor_smoke_name = "14B_cloud_advisor_smoke"

if not os.environ.get("OPENAI_API_KEY", "").strip():
    raise RuntimeError("OPENAI_API_KEY is not set even after ensure_openai_api_key().")

try:
    out = run_ga_all_categories(
        model_key="qwen25_coder_14b",
        categories=(1,),
        limit_per_category=1,
        sample_size=1,
        validation_size=1,
        population=2,
        gens=1,
        target_detpass=90,
        base_prefix=f"smoke_advisor_{SERVER_PRESET}",
        use_advisor=True,
        full_run=True,
        progress="verbose",
        timeout_sec=1200,
        retries=0,
        advisor_trigger_mode="always",
        advisor_min_population_for_child=2,
        advisor_force_child_quota=True,
        use_mock_advisor=False,
    )

    smoke_results[advisor_smoke_name] = out
    row = summarize_ga_output(advisor_smoke_name, out)
    smoke_rows.append(row)

    print(f"[PASS] {advisor_smoke_name}: {out}")
    print("[SUMMARY]", row)

except Exception as e:
    print(f"[FAIL] {advisor_smoke_name}: {type(e).__name__}: {e}")
    traceback.print_exc()
    smoke_results[advisor_smoke_name] = None
    smoke_rows.append({
        "run": advisor_smoke_name,
        "ok": False,
        "out_dir": None,
        "error": repr(e),
    })


# ============================================================
# 7. FINAL REPORT
# ============================================================

print("\n" + "=" * 120)
print("[FINAL SMOKE REPORT]")
print("=" * 120)

smoke_df = pd.DataFrame(smoke_rows)
display(smoke_df)

required_smoke_names = [f"{label}_cloudless_smoke" for label, _ in SMOKE_MODEL_LIST]
failed_required = []

for name in required_smoke_names:
    row = smoke_df[smoke_df["run"] == name]
    if len(row) == 0 or not bool(row.iloc[0].get("ok", False)):
        failed_required.append(name)

advisor_row = smoke_df[smoke_df["run"] == advisor_smoke_name]

advisor_ok = False
if len(advisor_row) > 0:
    ar = advisor_row.iloc[0]
    advisor_ok = (
        bool(ar.get("ok", False))
        and int(ar.get("advisor_proposal_count", 0) or 0) > 0
        and "advisor" in str(ar.get("proposal_sources", ""))
    )

if failed_required:
    print("[DO NOT RUN FULL FAIR LOOP]")
    print("Failed required cloudless smoke tests:", failed_required)

    for name in failed_required:
        out = smoke_results.get(name)
        if out:
            print("\n" + "=" * 120)
            print(f"[DEBUG CANDIDATES] {name}")
            print("=" * 120)
            print_candidate_error_tail(out)

    raise RuntimeError(f"Required cloudless smoke tests failed: {failed_required}")

if not advisor_ok:
    print("[DO NOT RUN CLOUD_ADVISOR FULL LOOP]")
    print("Cloud advisor smoke failed. Check OPENAI_API_KEY, advisor response files, and feedback guard.")
    out = smoke_results.get(advisor_smoke_name)
    if out:
        print_candidate_error_tail(out)
    raise RuntimeError("Cloud advisor smoke failed.")

print("[OK] Required cloudless smoke tests passed.")
print("[OK] Cloud advisor smoke passed.")
print("READY_FOR_FULL_FAIR_LOOP = True")

smoke_results

[OK] OPENAI_API_KEY is already set.

[CONFIG]
REPO: /root/llm/JOILang-Server True
SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py True
PYTHON: /root/llm/je/bin/python True
LOCAL_MODEL_BASE: /root/llm/local_models True

[PATCH CHECK] feedback=None guard
[OK] feedback=None guard already exists.
[OK] py_compile passed.

[MODEL PATH CHECK]


,model_key,path,exists,config,tokenizer,index,safetensors_count,size
0,qwen25_coder_7b,/root/llm/local_models/qwen25_coder_7b,True,True,True,True,4,15G
1,llama31_8b,/root/llm/local_models/llama31_8b,True,True,True,True,4,30G
2,qwen25_coder_14b,/root/llm/local_models/qwen25_coder_14b,True,True,True,True,6,28G
3,phi35_mini,/root/llm/local_models/phi35_mini,True,True,True,True,2,7.2G
4,gemma2_9b_it,/root/llm/local_models/gemma2_9b_it,True,True,True,True,4,18G


[OK] Required smoke models are available: ['qwen25_coder_7b', 'llama31_8b', 'qwen25_coder_14b']

[SMOKE RUNS] cloudless

########################################################################################################################
START 7B_cloudless_smoke: qwen25_coder_7b
########################################################################################################################
[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: qwen25_coder_7b / cloudless
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_pathcheck_A100_SET_B_cloudless_cat1_lpc1_pop1_gens1_qwen25_coder_7b_20260609_205337/ga_output
LAUNCHER_PYTHON: /root/llm/je/bin/python
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/local_models
JOI_V15_LOCAL_MODEL_NAME: /root/llm/local_models/qwen25_coder_7b
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DE

,run,ok,out_dir,summary_exists,progress_exists,best_DETPass,best_so_far_DETPass,accepted_best_DETPass,validation_det_pass_rate,validation_avg_det_score,...,tokens_gt_0,advisor_used,cloudless_mutation_used,advisor_proposal_count,proposal_sources,stop_reason,final_phase,generation_phase,next_action,plateau_type
0,7B_cloudless_smoke,True,/root/llm/JOILang-Server/gpt_mg/version0_15_up...,True,True,100.0,100.0,NaN,100.0,100.0,...,True,False,True,0,cloudless,max generations reached,FINAL_SELECTION,FINAL_SELECTION,stop_and_finalize,max_generation_reached
1,8B_cloudless_smoke,True,/root/llm/JOILang-Server/gpt_mg/version0_15_up...,True,True,100.0,100.0,NaN,100.0,100.0,...,True,False,True,0,cloudless,max generations reached,FINAL_SELECTION,FINAL_SELECTION,stop_and_finalize,max_generation_reached
2,14B_cloudless_smoke,True,/root/llm/JOILang-Server/gpt_mg/version0_15_up...,True,True,100.0,100.0,NaN,100.0,100.0,...,True,False,True,0,cloudless,max generations reached,FINAL_SELECTION,FINAL_SELECTION,stop_and_finalize,max_generation_reached
3,14B_cloud_advisor_smoke,True,/root/llm/JOILang-Server/gpt_mg/version0_15_up...,True,True,100.0,100.0,NaN,100.0,100.0,...,True,False,True,1,"advisor,cloudless",max generations reached,FINAL_SELECTION,FINAL_SELECTION,stop_and_finalize,max_generation_reached


[OK] Required cloudless smoke tests passed.
[OK] Cloud advisor smoke passed.
READY_FOR_FULL_FAIR_LOOP = True


{'7B_cloudless_smoke': PosixPath('/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_pathcheck_A100_SET_B_cloudless_cat1_lpc1_pop1_gens1_qwen25_coder_7b_20260609_205337/ga_output'),
 '8B_cloudless_smoke': PosixPath('/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_pathcheck_A100_SET_B_cloudless_cat1_lpc1_pop1_gens1_llama31_8b_20260609_205421/ga_output'),
 '14B_cloudless_smoke': PosixPath('/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_pathcheck_A100_SET_B_cloudless_cat1_lpc1_pop1_gens1_qwen25_coder_14b_20260609_205515/ga_output'),
 '14B_cloud_advisor_smoke': PosixPath('/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_advisor_A100_SET_B_cloud_advisor_cat1_lpc1_pop2_gens1_qwen25_coder_14b_20260609_205639/ga_output')}

## Cell 3. 3개 모델 × 2조건 자동 실행

### 본격 6개 run 전에 반드시 1회만 테스트

In [27]:
sanity_out = run_ga_all_categories(
    model_key="qwen25_coder_7b",
    categories=range(1, 3),
    limit_per_category=1,
    sample_size=2,
    validation_size=2,
    population=2,
    gens=1,
    target_detpass=90,
    base_prefix=f"sanity_{SERVER_PRESET}",
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=600,
    retries=0,
)

sanity_out

[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: qwen25_coder_7b / cloudless
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/sanity_A100_SET_B_cloudless_cat12_lpc1_pop2_gens1_qwen25_coder_7b_20260602_222319/ga_output
LAUNCHER_PYTHON: /root/llm/je/bin/python
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_cloudless_20260602_222319.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 90 --population 2 --gens 1 --min-generations 1 --max-generations 1 --sample-size 2 --validation-size 2 --cheap-eval-limit 2 --candidate-k 1 --repair-attempts 0 --det-profile strict --feedback-gui

PosixPath('/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/sanity_A100_SET_B_cloudless_cat12_lpc1_pop2_gens1_qwen25_coder_7b_20260602_222319/ga_output')

In [34]:
ga_runs_fair = {}

for label, model_key in MODEL_LIST:
    for mode_name, use_advisor in RUN_MODES:
        run_key = f"{label}_{mode_name}"

        print("\n" + "#" * 120)
        print(f"START RUN: {run_key} / {model_key}")
        print("#" * 120)

        try:
            out_dir = run_ga_all_categories(
                model_key=model_key,
                categories=COMMON_GA_CONFIG["categories"],
                limit_per_category=COMMON_GA_CONFIG["limit_per_category"],
                sample_size=COMMON_GA_CONFIG["sample_size"],
                validation_size=COMMON_GA_CONFIG["validation_size"],
                population=COMMON_GA_CONFIG["population"],
                gens=COMMON_GA_CONFIG["gens"],
                target_detpass=COMMON_GA_CONFIG["target_detpass"],
                base_prefix=f"ga_{SERVER_PRESET}",
                use_advisor=use_advisor,
                full_run=COMMON_GA_CONFIG["full_run"],
                progress=COMMON_GA_CONFIG["progress"],
                timeout_sec=COMMON_GA_CONFIG["timeout_sec"],
                retries=COMMON_GA_CONFIG["retries"],
                idle_timeout_sec=COMMON_GA_CONFIG["idle_timeout_sec"],
                total_timeout_sec=COMMON_GA_CONFIG["total_timeout_sec"],

                advisor_trigger_mode="always",
                advisor_min_population_for_child=4,
                advisor_force_child_quota=True,
                use_mock_advisor=False,
            )

            ga_runs_fair[run_key] = out_dir
            print(f"[PASS] {run_key}: {out_dir}")

        except Exception as e:
            print(f"[FAIL] {run_key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            ga_runs_fair[run_key] = None

        time.sleep(5)


valid_ga_runs_fair = {
    key: path
    for key, path in ga_runs_fair.items()
    if path is not None
}

valid_ga_runs_fair


########################################################################################################################
START RUN: 7B_cloudless / qwen25_coder_7b
########################################################################################################################
[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: qwen25_coder_7b / cloudless
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A100_SET_B_cloudless_cat12345678_lpc3_pop5_gens10_qwen25_coder_7b_20260603_202722/ga_output
LAUNCHER_PYTHON: /root/llm/je/bin/python
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_cloudless_20260603_202722.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413

Traceback (most recent call last):
  File "/tmp/ipykernel_1604739/2440488765.py", line 12, in <module>
    out_dir = run_ga_all_categories(
  File "/tmp/ipykernel_1604739/2989200631.py", line 210, in run_ga_all_categories
    raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")
RuntimeError: run_ga_all_categories failed with return code 1



########################################################################################################################
START RUN: 7B_cloud_advisor / qwen25_coder_7b
########################################################################################################################
[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: qwen25_coder_7b / cloud_advisor
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A100_SET_B_cloud_advisor_cat12345678_lpc3_pop5_gens10_qwen25_coder_7b_20260603_204713/ga_output
LAUNCHER_PYTHON: /root/llm/je/bin/python
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_cloud_advisor_20260603_204713.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_1

Traceback (most recent call last):
  File "/tmp/ipykernel_1604739/2440488765.py", line 12, in <module>
    out_dir = run_ga_all_categories(
  File "/tmp/ipykernel_1604739/2989200631.py", line 210, in run_ga_all_categories
    raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")
RuntimeError: run_ga_all_categories failed with return code 1



########################################################################################################################
START RUN: 8B_cloudless / llama31_8b
########################################################################################################################
[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: llama31_8b / cloudless
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A100_SET_B_cloudless_cat12345678_lpc3_pop5_gens10_llama31_8b_20260603_210702/ga_output
LAUNCHER_PYTHON: /root/llm/je/bin/python
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_llama31_8b_cloudless_20260603_210702.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_sear

...


########################################################################################################################
START RUN: 14B_cloudless / qwen25_coder_14b
########################################################################################################################
[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: qwen25_coder_14b / cloudless
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A100_SET_B_cloudless_cat12345678_lpc3_pop5_gens10_qwen25_coder_14b_20260603_214645/ga_output
LAUNCHER_PYTHON: /root/llm/je/bin/python
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_14b_cloudless_20260603_214645.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update202

Traceback (most recent call last):
  File "/tmp/ipykernel_1604739/2440488765.py", line 12, in <module>
    out_dir = run_ga_all_categories(
  File "/tmp/ipykernel_1604739/2989200631.py", line 210, in run_ga_all_categories
    raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")
RuntimeError: run_ga_all_categories failed with return code 1



########################################################################################################################
START RUN: 14B_cloud_advisor / qwen25_coder_14b
########################################################################################################################
[SKIP unsupported flag] --idle-timeout-sec
[SKIP unsupported flag] --total-timeout-sec
RUN: qwen25_coder_14b / cloud_advisor
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_A100_SET_B_cloud_advisor_cat12345678_lpc3_pop5_gens10_qwen25_coder_14b_20260603_220649/ga_output
LAUNCHER_PYTHON: /root/llm/je/bin/python
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_14b_cloud_advisor_20260603_220649.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/versi

Traceback (most recent call last):
  File "/tmp/ipykernel_1604739/2440488765.py", line 12, in <module>
    out_dir = run_ga_all_categories(
  File "/tmp/ipykernel_1604739/2989200631.py", line 210, in run_ga_all_categories
    raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")
RuntimeError: run_ga_all_categories failed with return code 1


{}

## Cell 4. 결과 요약표 생성


In [35]:
def load_json_safe(path: Path, default=None):
    if default is None:
        default = {}
    try:
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    except Exception:
        pass
    return default


def read_csv_safe(path: Path):
    try:
        if path.exists() and path.stat().st_size > 0:
            return pd.read_csv(path)
    except Exception:
        pass
    return pd.DataFrame()


def summarize_ga_run(run_key: str, out_dir):
    out_dir = Path(out_dir)

    summary = load_json_safe(out_dir / "ga_summary.json", default={})
    progress_df = read_csv_safe(out_dir / "ga_generation_progress.csv")
    diag_df = read_csv_safe(out_dir / "ga_population_diagnostics.csv")

    last_progress = {}
    if not progress_df.empty:
        last_progress = progress_df.tail(1).iloc[0].to_dict()

    failure_histogram = None
    if not diag_df.empty and "failure_histogram" in diag_df.columns:
        failure_histogram = diag_df.tail(1)["failure_histogram"].iloc[0]

    label, mode = run_key.split("_", 1)

    return {
        "server_preset": SERVER_PRESET,
        "run_key": run_key,
        "label": label,
        "mode": mode,
        "out_dir": str(out_dir),

        "target_detpass": COMMON_GA_CONFIG["target_detpass"],
        "categories": ",".join(map(str, COMMON_GA_CONFIG["categories"])),
        "limit_per_category": COMMON_GA_CONFIG["limit_per_category"],
        "sample_size": COMMON_GA_CONFIG["sample_size"],
        "validation_size": COMMON_GA_CONFIG["validation_size"],
        "population": COMMON_GA_CONFIG["population"],
        "gens": COMMON_GA_CONFIG["gens"],

        "stage": summary.get("stage"),
        "stop_reason": summary.get("stop_reason"),
        "best_genome_id": summary.get("best_genome_id"),
        "best_generation": summary.get("best_generation"),

        "best_DETPass": summary.get("best_DETPass"),
        "best_avg_DET": summary.get("best_avg_DET"),
        "accepted_best_DETPass": summary.get("accepted_best_DETPass"),
        "accepted_best_avg_DET": summary.get("accepted_best_avg_DET"),
        "accepted_best_tokens": summary.get("accepted_best_tokens"),

        "compact_best_eligible": summary.get("compact_best_eligible"),
        "compact_best_DETPass": summary.get("compact_best_DETPass"),
        "compact_best_tokens": summary.get("compact_best_tokens"),

        "advisor_status": summary.get("advisor_status"),
        "advisor_used": summary.get("advisor_used"),
        "advisor_proposals_generated": summary.get("advisor_proposals_generated"),
        "advisor_proposals_accepted_applied": summary.get("advisor_proposals_accepted_applied"),
        "advisor_proposals_rejected": summary.get("advisor_proposals_rejected"),
        "advisor_children_scheduled": summary.get("advisor_children_scheduled"),

        "cloudless_mutation_used": summary.get("cloudless_mutation_used"),
        "pareto_archive_size": summary.get("pareto_archive_size"),

        "last_progress_fitness": last_progress.get("fitness"),
        "last_progress_avg_det_score": last_progress.get("avg_det_score"),
        "last_progress_det_pass_rate": last_progress.get("det_pass_rate"),
        "last_progress_advisor_triggered": last_progress.get("advisor_triggered"),

        "failure_histogram": failure_histogram,

        "summary_exists": (out_dir / "ga_summary.json").exists(),
        "progress_exists": (out_dir / "ga_generation_progress.csv").exists(),
    }


summary_rows = []

for run_key, out_dir in ga_runs_fair.items():
    if out_dir is None:
        label, mode = run_key.split("_", 1)
        summary_rows.append({
            "server_preset": SERVER_PRESET,
            "run_key": run_key,
            "label": label,
            "mode": mode,
            "run_status": "failed",
            "out_dir": None,
        })
    else:
        row = summarize_ga_run(run_key, out_dir)
        row["run_status"] = "success"
        summary_rows.append(row)

fair_summary_df = pd.DataFrame(summary_rows)

summary_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_fair_summary.csv"
fair_summary_df.to_csv(summary_csv, index=False)

print("summary_csv:", summary_csv)
display(fair_summary_df)

summary_csv: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/fair_compare_A100_SET_B_20260602_222041/A100_SET_B_fair_summary.csv


,server_preset,run_key,label,mode,run_status,out_dir
0,A100_SET_B,7B_cloudless,7B,cloudless,failed,None
1,A100_SET_B,7B_cloud_advisor,7B,cloud_advisor,failed,None
2,A100_SET_B,8B_cloudless,8B,cloudless,failed,None
3,A100_SET_B,8B_cloud_advisor,8B,cloud_advisor,failed,None
4,A100_SET_B,14B_cloudless,14B,cloudless,failed,None
5,A100_SET_B,14B_cloud_advisor,14B,cloud_advisor,failed,None


## Cell 5. cloudless vs cloud-advisor delta 표 생성


In [36]:
def make_delta_table(fair_summary_df):
    df = fair_summary_df.copy()
    df = df[df["run_status"] == "success"].copy()

    rows = []

    for label in ["7B", "8B", "14B"]:
        c = df[(df["label"] == label) & (df["mode"] == "cloudless")]
        a = df[(df["label"] == label) & (df["mode"] == "cloud_advisor")]

        if c.empty or a.empty:
            rows.append({
                "server_preset": SERVER_PRESET,
                "label": label,
                "status": "missing_pair",
            })
            continue

        c = c.iloc[0]
        a = a.iloc[0]

        row = {
            "server_preset": SERVER_PRESET,
            "label": label,
            "status": "ok",

            "cloudless_best_DETPass": c.get("best_DETPass"),
            "advisor_best_DETPass": a.get("best_DETPass"),
            "delta_best_DETPass": None,

            "cloudless_best_avg_DET": c.get("best_avg_DET"),
            "advisor_best_avg_DET": a.get("best_avg_DET"),
            "delta_best_avg_DET": None,

            "cloudless_tokens": c.get("accepted_best_tokens"),
            "advisor_tokens": a.get("accepted_best_tokens"),
            "delta_tokens": None,

            "cloudless_best_generation": c.get("best_generation"),
            "advisor_best_generation": a.get("best_generation"),

            "advisor_used": a.get("advisor_used"),
            "advisor_proposals_generated": a.get("advisor_proposals_generated"),
            "advisor_proposals_accepted_applied": a.get("advisor_proposals_accepted_applied"),
            "advisor_children_scheduled": a.get("advisor_children_scheduled"),

            "cloudless_out_dir": c.get("out_dir"),
            "advisor_out_dir": a.get("out_dir"),
        }

        for out_key, adv_key, base_key in [
            ("delta_best_DETPass", "advisor_best_DETPass", "cloudless_best_DETPass"),
            ("delta_best_avg_DET", "advisor_best_avg_DET", "cloudless_best_avg_DET"),
            ("delta_tokens", "advisor_tokens", "cloudless_tokens"),
        ]:
            try:
                row[out_key] = row[adv_key] - row[base_key]
            except Exception:
                row[out_key] = None

        rows.append(row)

    delta_df = pd.DataFrame(rows)

    delta_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_cloudless_vs_advisor_delta.csv"
    delta_df.to_csv(delta_csv, index=False)

    print("delta_csv:", delta_csv)
    display(delta_df)

    return delta_df


fair_delta_df = make_delta_table(fair_summary_df)

delta_csv: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/fair_compare_A100_SET_B_20260602_222041/A100_SET_B_cloudless_vs_advisor_delta.csv


,server_preset,label,status
0,A100_SET_B,7B,missing_pair
1,A100_SET_B,8B,missing_pair
2,A100_SET_B,14B,missing_pair


## Cell 6. advisor activity 확인

In [37]:
def inspect_advisor_activity(ga_runs_fair):
    rows = []

    for run_key, out_dir in ga_runs_fair.items():
        if out_dir is None:
            continue

        out_dir = Path(out_dir)

        advisor_feedback = out_dir / "advisor_feedback_batches.jsonl"
        advisor_proposals = out_dir / "advisor_mutation_proposals.jsonl"
        advisor_summary = out_dir / "advisor_mutation_summary.csv"

        rows.append({
            "server_preset": SERVER_PRESET,
            "run_key": run_key,
            "out_dir": str(out_dir),
            "advisor_feedback_exists": advisor_feedback.exists(),
            "advisor_feedback_bytes": advisor_feedback.stat().st_size if advisor_feedback.exists() else 0,
            "advisor_proposals_exists": advisor_proposals.exists(),
            "advisor_proposals_bytes": advisor_proposals.stat().st_size if advisor_proposals.exists() else 0,
            "advisor_summary_exists": advisor_summary.exists(),
            "advisor_summary_bytes": advisor_summary.stat().st_size if advisor_summary.exists() else 0,
        })

    advisor_df = pd.DataFrame(rows)

    advisor_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_advisor_activity.csv"
    advisor_df.to_csv(advisor_csv, index=False)

    print("advisor_csv:", advisor_csv)
    display(advisor_df)

    return advisor_df


advisor_activity_df = inspect_advisor_activity(ga_runs_fair)

advisor_csv: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/fair_compare_A100_SET_B_20260602_222041/A100_SET_B_advisor_activity.csv


""
